# T01: Getting Started

This tutorial introduces the undata system. You will:
- Confirm the backend service is healthy
- List available schema sources
- Browse the first few schema elements
- Verify API key authentication

**Services required**: backend (`http://localhost:8002`)

**Est. time**: 5 min

In [1]:
# Cell 2 — service availability check
import os

import httpx

BACKEND_URL = os.getenv("BACKEND_URL", "http://localhost:8002")
API_KEY = os.getenv(
    "API_KEY",
    "qs005testtoken1234567890abcdef1234567890abcdef1234567890abcdef12",
)
HEADERS = {"Authorization": f"Bearer {API_KEY}"}

try:
    httpx.get(f"{BACKEND_URL}/health", timeout=2.0).raise_for_status()
    print(f"✓ Backend available at {BACKEND_URL}")
except Exception as _e:
    import pytest

    pytest.skip(f"Backend unavailable: {_e}")

✓ Backend available at http://localhost:8002


## 1. Health Check

The backend exposes a `/health` endpoint that returns a simple status object.
This is the canonical way to confirm the service is up.

In [2]:
response = httpx.get(f"{BACKEND_URL}/health", timeout=5.0)
assert response.status_code == 200, f"Expected 200, got {response.status_code}"
data = response.json()
assert data["status"] == "ok", f"Expected status=ok, got {data}"
print(f"Health response: {data}")

Health response: {'status': 'ok', 'version': '2026.03.0'}


## 2. List Schema Sources

Schema sources represent the origin of schema elements (e.g., BIDS, DANDI, NWB).
Each source has a unique name and format. The `/api/v1/sources/` endpoint lists all
registered sources.

In [3]:
response = httpx.get(f"{BACKEND_URL}/api/v1/sources/", headers=HEADERS, timeout=5.0)
assert response.status_code == 200, f"Expected 200, got {response.status_code}"
data = response.json()
print(f"Found {len(data['items'])} sources")
for source in data["items"]:
    print(f"  - {source['name']} (format: {source.get('format', 'unknown')})")

Found 9 sources
  - BIDS (format: yaml)
  - DANDI (format: json)
  - QS005DbgSrc1773235766 (format: json)
  - QS005Perf1773236575 (format: json)
  - QS005Perf1773236867 (format: json)
  - QS005Src1773235717 (format: json)
  - QS005Src1773235974 (format: json)
  - QS005Src-fix (format: json)
  - undata (format: canonical)


## 3. List Elements (first 5)

Schema elements are the atomic units — individual fields, slots, or variables defined
across all ingested schemas. The `/api/v1/elements/` endpoint supports pagination
and filtering.

In [4]:
response = httpx.get(
    f"{BACKEND_URL}/api/v1/elements/",
    headers=HEADERS,
    params={"limit": 5},
    timeout=5.0,
)
assert response.status_code == 200, f"Expected 200, got {response.status_code}"
data = response.json()
print(f"Total elements in backend: {data['total']}")
print("First 5:")
for item in data["items"]:
    print(f"  - {item['name']} | type={item['data_type']} | source={item.get('source_name', '?')}")

Total elements in backend: 1417
First 5:
  - url | type=string | source=?
  - relation | type=string | source=?
  - identifier | type=string | source=?
  - name | type=string | source=?
  - id | type=string | source=?


## 4. Authenticate — About API Keys

The backend uses Bearer token authentication. Each request must include an
`Authorization: Bearer <token>` header.

In development, set the `API_KEY` environment variable to a valid token
from your backend. The default uses the quickstart-test dev token (seeded
by the backend quickstart setup). To get a token for a fresh backend,
authenticate via Keycloak and issue an API key:

```bash
# Issue an API key via the backend API (requires Keycloak OIDC token first)
POST /api/v1/tokens/

# Or set the env var directly:
export API_KEY=<your-token>
```

In [5]:
# Verify authentication works with the current token
response = httpx.get(f"{BACKEND_URL}/api/v1/users/me", headers=HEADERS, timeout=5.0)
if response.status_code == 200:
    user = response.json()
    name = user.get("display_name", user.get("email", user.get("id", "?")))
    print(f"✓ Authenticated as: {name}")
    print(f"  User ID: {user.get('id', '?')}")
    print(f"  Roles:   {user.get('roles', [])}")
elif response.status_code == 401:
    print("⚠ Auth returned 401 — API_KEY may not be seeded in this backend.")
    print("  Set API_KEY env var to a valid bearer token and re-run.")
    print(f"  Current BACKEND_URL: {BACKEND_URL}")
    print("  To seed the quickstart test key, run the backend quickstart SQL:")
    print("  See backend/README.md or specs/002-schema-backend/quickstart.md")
else:
    print(f"⚠ Unexpected response: {response.status_code} — {response.text[:200]}")

✓ Authenticated as: QS005 Test User
  User ID: 11111111-1111-1111-1111-111111111111
  Roles:   ['curator']


## Next Steps

You've confirmed the backend is healthy, can list sources and elements, and that
authentication works.

Next: **[T02: Ingest Schemas via CLI](02_ingest_schemas.ipynb)** — push BIDS and DANDI
schema data into the backend using the `undata ingest` CLI commands.